# Задача 04. Повторные покупки и LTV покупателей

**Постановка задачи:** нужно оценить, сколько покупателей делают повторные покупки и какую выручку они приносят. Для каждого клиента посчитать:

- дату первого заказа;
- количество доставленных заказов;
- суммарную выручку клиента;
- сегмент `one_time` или `repeat`.

Затем агрегировать результат по сегментам.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH order_finance AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_date,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'delivered'
    GROUP BY o.order_id, o.customer_id, o.order_date
), customer_ltv AS (
    SELECT
        customer_id,
        MIN(order_date) AS first_order_date,
        COUNT(order_id) AS delivered_orders,
        SUM(revenue) AS ltv
    FROM order_finance
    GROUP BY customer_id
), customer_segments AS (
    SELECT
        customer_id,
        first_order_date,
        delivered_orders,
        ltv,
        CASE
            WHEN delivered_orders = 1 THEN 'one_time'
            ELSE 'repeat'
        END AS customer_segment
    FROM customer_ltv
)
SELECT
    customer_segment,
    COUNT(customer_id) AS customers,
    ROUND(AVG(delivered_orders), 2) AS avg_delivered_orders,
    ROUND(AVG(ltv), 2) AS avg_ltv,
    ROUND(SUM(ltv), 2) AS total_revenue
FROM customer_segments
GROUP BY customer_segment
ORDER BY total_revenue DESC;
"""

df = pd.read_sql_query(query, conn)
df

In [ ]:
# Дополнительно покажу 10 покупателей с самым высоким LTV
detail_query = r'''
WITH order_finance AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_date,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'delivered'
    GROUP BY o.order_id, o.customer_id, o.order_date
)
SELECT
    customer_id,
    MIN(order_date) AS first_order_date,
    COUNT(order_id) AS delivered_orders,
    ROUND(SUM(revenue), 2) AS ltv
FROM order_finance
GROUP BY customer_id
ORDER BY ltv DESC
LIMIT 10;
'''
pd.read_sql_query(detail_query, conn)